In [1]:
import fastf1
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

fastf1.Cache.enable_cache("f1_cache")

In [2]:
q = fastf1.get_session(2024, "Netherlands", "Qualifying")
q.load()  # loads quali data (laps, results, etc.)

# Best qualifying lap per driver (seconds) + Team
q_laps = q.laps.dropna(subset=["LapTime"]).copy()
best_q = (
    q_laps.sort_values("LapTime")
          .groupby("Driver", as_index=False)
          .first()[["Driver", "Team", "LapTime"]]
          .rename(columns={"LapTime": "QualifyingTime_s"})
)
best_q["QualifyingTime_s"] = best_q["QualifyingTime_s"].dt.total_seconds()

core           INFO 	Loading data for Dutch Grand Prix - Qualifying [v3.6.0]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	D

In [3]:
r = fastf1.get_session(2024, "Netherlands", "R")
r.load()

laps = r.laps.dropna(subset=["LapTime"]).copy()
# Keep “proper” race laps: drop pit in/out laps (simple, robust rule)
if "PitInLap" in laps.columns and "PitOutLap" in laps.columns:
    laps = laps[~(laps["PitInLap"] | laps["PitOutLap"])].copy()

laps["LapTime_s"] = laps["LapTime"].dt.total_seconds()

# Trim per-driver outliers (5th–95th percentile) to stabilize the target
def _trim(g):
    lo, hi = np.percentile(g["LapTime_s"], [5, 95])
    return g[(g["LapTime_s"] >= lo) & (g["LapTime_s"] <= hi)]

laps_trim = laps.groupby("Driver", group_keys=False).apply(_trim)

y_tbl = laps_trim.groupby("Driver", as_index=False)["LapTime_s"].mean()
y_tbl = y_tbl.rename(columns={"LapTime_s": "TargetMeanRaceLap_s"})

core           INFO 	Loading data for Dutch Grand Prix - Race [v3.6.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['4', '1', '16', '81', '55', '11', '63', '44', '10', '14', '27', '3', '18', '23', '31', '2', '22', '20', '77', '24']
C:\Users\musab\AppData\Local\Temp\ipykernel_17156\1500392236.py:

In [4]:
train_df = best_q.merge(y_tbl, on="Driver", how="inner")

# Features: ONLY qualifying time (+ optional Team one-hot). No weather.
feature_cols_num = ["QualifyingTime_s"]
feature_cols_cat = ["Team"]  # optional but useful
X = train_df[feature_cols_num + feature_cols_cat].copy()
y = train_df["TargetMeanRaceLap_s"].copy()

In [6]:
num_tf = Pipeline([("imp", SimpleImputer(strategy="median"))])
cat_tf = Pipeline([
    ("imp", SimpleImputer(strategy="most_frequent")),
    ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

pre = ColumnTransformer([
    ("num", num_tf, feature_cols_num),
    ("cat", cat_tf, feature_cols_cat),
])

model = Pipeline([
    ("pre", pre),
    ("gbr", GradientBoostingRegressor(
        n_estimators=400, learning_rate=0.05, max_depth=3, random_state=42
    )),
])

# Small train/test split across drivers (it’s still one event, but fine for a demo)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=7)
model.fit(X_tr, y_tr)
y_hat = model.predict(X_te)
print(f"MAE on 2024 Dutch (driver hold-out): {mean_absolute_error(y_te, y_hat):.3f} s")

MAE on 2024 Dutch (driver hold-out): 0.487 s


In [11]:
# q2025 = fastf1.get_session(2025, "Netherlands", "Q")  # "Q" or "Qualifying" both work (This will work on August 30, 2025)
# q2025.load()

# Manual data entry - create a mock q2025 object with results attribute
import pandas as pd

class MockSession:
    def __init__(self):
        # Use seconds format for easier conversion
        self.results = pd.DataFrame({
            "Abbreviation": ["VER", "NOR", "PIA", "RUS", "LEC", "HAM", "SAI", "PER", "ALB", "HUL"],
            "TeamName": ["Red Bull Racing Honda RBPT", "McLaren Mercedes", "McLaren Mercedes", 
                        "Mercedes", "Ferrari", "Mercedes", "Ferrari", "Red Bull Racing Honda RBPT",
                        "Williams Mercedes", "Haas Ferrari"],
            # Times in seconds (1:10.567 = 70.567 seconds)
            "Q1": ["70.567", "70.789", "70.892", "70.945", "71.123", 
                   "71.234", "71.345", "71.456", "71.567", "71.678"],
            "Q2": ["70.123", "70.234", "70.345", "70.456", "70.567",
                   "70.678", "70.789", "70.890", "70.991", "71.092"],
            "Q3": ["69.789", "69.890", "69.991", "70.092", "70.193",
                   "70.294", "70.395", "70.496", None, None]  # Q3 times only for top 10
        })

q2025 = MockSession()

# Now your existing code will work exactly the same:
# results has columns like: Abbreviation (driver code), TeamName, Q1, Q2, Q3 (strings)
res25 = q2025.results[["Abbreviation", "TeamName", "Q1", "Q2", "Q3"]].copy()

# convert Q1/Q2/Q3 to timedeltas, then seconds
for col in ["Q1", "Q2", "Q3"]:
    # First convert to numeric (seconds), then to timedelta
    res25[col] = pd.to_numeric(res25[col], errors="coerce")
    res25[col] = pd.to_timedelta(res25[col], unit='s')

res25["QualifyingTime_s"] = res25[["Q1","Q2","Q3"]].min(axis=1).dt.total_seconds()

inf_2025 = res25.rename(columns={"Abbreviation":"Driver", "TeamName":"Team"})[["Driver","Team","QualifyingTime_s"]]

# (Optional) filter to drivers you trained on (if needed)
# inf_2025 = inf_2025[inf_2025["Driver"].isin(train_df["Driver"].unique())]

# predict
X_2025 = inf_2025[feature_cols_num + feature_cols_cat].copy()  # same cols/order as training
pred_2025 = model.predict(X_2025)

out_2025 = inf_2025[["Driver","Team","QualifyingTime_s"]].copy()
out_2025["PredictedMeanRaceLap_s"] = pred_2025
out_2025 = out_2025.sort_values("PredictedMeanRaceLap_s").reset_index(drop=True)

print("\n=== Dutch 2025 — predicted race pace (lower is faster) ===")
print(out_2025)


=== Dutch 2025 — predicted race pace (lower is faster) ===
  Driver                        Team  QualifyingTime_s  PredictedMeanRaceLap_s
0    VER  Red Bull Racing Honda RBPT            69.789               75.295849
1    NOR            McLaren Mercedes            69.890               75.556157
2    PIA            McLaren Mercedes            69.991               75.556157
3    RUS                    Mercedes            70.092               75.565839
4    SAI                     Ferrari            70.395               75.625666
5    LEC                     Ferrari            70.193               75.628105
6    PER  Red Bull Racing Honda RBPT            70.496               75.685031
7    HAM                    Mercedes            70.294               75.689699
8    ALB           Williams Mercedes            70.991               77.002363
9    HUL                Haas Ferrari            71.092               77.002363


In [12]:
# race25 = fastf1.get_session(2025, "Netherlands", "R")
# race25.load()

# # Remove pit in/out laps and safety car laps
# laps = race25.laps.dropna(subset=["LapTime"]).copy()
# if "PitInLap" in laps.columns and "PitOutLap" in laps.columns:
#     laps = laps[~(laps["PitInLap"] | laps["PitOutLap"])]

# # Remove laps under VSC/SC if info available
# if "TrackStatus" in laps.columns:
#     laps = laps[~laps["TrackStatus"].astype(str).str.contains("4|5", na=False)]  # 4=SC, 5=VSC

# laps["LapTime_s"] = laps["LapTime"].dt.total_seconds()

# # Compute actual median race pace
# actual_pace = laps.groupby("Driver", as_index=False)["LapTime_s"].median()
# actual_pace = actual_pace.rename(columns={"LapTime_s": "ActualMedianRaceLap_s"})

# # Merge with prediction
# compare = out_2025.merge(actual_pace, on="Driver", how="left")
# compare["Error_s"] = compare["PredictedMeanRaceLap_s"] - compare["ActualMedianRaceLap_s"]

# # Sort by actual pace
# compare = compare.sort_values("ActualMedianRaceLap_s").reset_index(drop=True)

# print(compare)

# Since the 2025 Dutch GP hasn't happened yet, we can only show predictions
print("\n=== Dutch 2025 — predicted race pace (lower is faster) ===")
print("Note: Race hasn't occurred yet, so no actual times available for comparison")
print(out_2025)


=== Dutch 2025 — predicted race pace (lower is faster) ===
Note: Race hasn't occurred yet, so no actual times available for comparison
  Driver                        Team  QualifyingTime_s  PredictedMeanRaceLap_s
0    VER  Red Bull Racing Honda RBPT            69.789               75.295849
1    NOR            McLaren Mercedes            69.890               75.556157
2    PIA            McLaren Mercedes            69.991               75.556157
3    RUS                    Mercedes            70.092               75.565839
4    SAI                     Ferrari            70.395               75.625666
5    LEC                     Ferrari            70.193               75.628105
6    PER  Red Bull Racing Honda RBPT            70.496               75.685031
7    HAM                    Mercedes            70.294               75.689699
8    ALB           Williams Mercedes            70.991               77.002363
9    HUL                Haas Ferrari            71.092               77.00